In [ ]:
!git clone https://github.com/facebookresearch/sam-3d-body.git

In [ ]:
cd /kaggle/working/sam-3d-body

In [ ]:
ls

In [ ]:
!pip install pytorch-lightning pyrender opencv-python yacs scikit-image einops timm dill pandas rich hydra-core hydra-submitit-launcher hydra-colorlog pyrootutils webdataset chump networkx==3.2.1 roma joblib seaborn wandb appdirs appnope ffmpeg cython jsonlines pytest xtcocotools loguru optree fvcore black pycocotools tensorboard huggingface_hub

In [ ]:
!pip install 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps

In [ ]:
!pip install git+https://github.com/microsoft/MoGe.git

In [ ]:
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("hf_token")
from huggingface_hub import login
login(token=hf_token)


In [ ]:
cd /kaggle/working/sam-3d-body/tools

In [ ]:
ls

In [ ]:
!mv vis_utils.py vis_utils_old.py

In [ ]:
ls

# *Left Hand Alignemnt with Right shifting*

How to tune:

First run with GLOBAL_CAM_TX = GLOBAL_CAM_TY = GLOBAL_CAM_TZ = 0.0 and FOCAL_LENGTH_SCALE = 1.0.
If the whole mesh is consistently shifted left/right/up/down, adjust GLOBAL_CAM_TX / GLOBAL_CAM_TY in small steps (e.g. ±0.01).
If the mesh is too big or too small, change FOCAL_LENGTH_SCALE (e.g. 0.9, 1.1, 1.3, …).
Both hands will always move together, so improving alignment for one hand won’t break the other anymore.

In [ ]:
%%writefile vis_utils.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
import numpy as np
import cv2
from sam_3d_body.visualization.renderer import Renderer
from sam_3d_body.visualization.skeleton_visualizer import SkeletonVisualizer
from sam_3d_body.metadata.mhr70 import pose_info as mhr70_pose_info

LIGHT_BLUE = (0.65098039, 0.74117647, 0.85882353)

# === ADJUST THIS VALUE TO CONTROL LEFTWARD SHIFT ===
# Positive value = shift mesh left (counteracts rightward misalignment)
# Try values like 0.05, 0.08, 0.10, 0.12, etc.
LEFT_SHIFT_AMOUNT = 0.015

visualizer = SkeletonVisualizer(line_width=2, radius=5)
visualizer.set_pose_meta(mhr70_pose_info)


def visualize_sample(img_cv2, outputs, faces):
    img_keypoints = img_cv2.copy()
    img_mesh = img_cv2.copy()

    rend_img = []
    for pid, person_output in enumerate(outputs):
        keypoints_2d = person_output["pred_keypoints_2d"]
        keypoints_2d = np.concatenate(
            [keypoints_2d, np.ones((keypoints_2d.shape[0], 1))], axis=-1
        )
        img1 = visualizer.draw_skeleton(img_keypoints.copy(), keypoints_2d)

        img1 = cv2.rectangle(
            img1,
            (int(person_output["bbox"][0]), int(person_output["bbox"][1])),
            (int(person_output["bbox"][2]), int(person_output["bbox"][3])),
            (0, 255, 0),
            2,
        )

        if "lhand_bbox" in person_output:
            img1 = cv2.rectangle(
                img1,
                (
                    int(person_output["lhand_bbox"][0]),
                    int(person_output["lhand_bbox"][1]),
                ),
                (
                    int(person_output["lhand_bbox"][2]),
                    int(person_output["lhand_bbox"][3]),
                ),
                (255, 0, 0),
                2,
            )

        if "rhand_bbox" in person_output:
            img1 = cv2.rectangle(
                img1,
                (
                    int(person_output["rhand_bbox"][0]),
                    int(person_output["rhand_bbox"][1]),
                ),
                (
                    int(person_output["rhand_bbox"][2]),
                    int(person_output["rhand_bbox"][3]),
                ),
                (0, 0, 255),
                2,
            )

        # === Apply left shift to camera translation ===
        adjusted_cam_t = person_output["pred_cam_t"].copy()
        adjusted_cam_t[0] -= LEFT_SHIFT_AMOUNT

        renderer = Renderer(focal_length=1.5 * person_output["focal_length"], faces=faces)
        
        # Front view overlay on original image
        img2 = (
            renderer(
                person_output["pred_vertices"],
                adjusted_cam_t,
                img_mesh.copy(),
                mesh_base_color=LIGHT_BLUE,
                scene_bg_color=(1, 1, 1),
            )
            * 255
        )

        # Side view on white background
        white_img = np.ones_like(img_cv2) * 255
        img3 = (
            renderer(
                person_output["pred_vertices"],
                adjusted_cam_t,
                white_img,
                mesh_base_color=LIGHT_BLUE,
                scene_bg_color=(1, 1, 1),
                side_view=True,
            )
            * 255
        )

        cur_img = np.concatenate([img_cv2, img1, img2, img3], axis=1)
        rend_img.append(cur_img)

    return rend_img


def visualize_sample_together(img_cv2, outputs, faces):
    # Render everything together
    img_keypoints = img_cv2.copy()
    img_mesh = img_cv2.copy()

    # Sort by depth (furthest to closest)
    all_depths = np.stack([tmp['pred_cam_t'] for tmp in outputs], axis=0)[:, 2]
    outputs_sorted = [outputs[idx] for idx in np.argsort(-all_depths)]

    # Draw all skeletons
    for person_output in outputs_sorted:
        keypoints_2d = person_output["pred_keypoints_2d"]
        keypoints_2d = np.concatenate(
            [keypoints_2d, np.ones((keypoints_2d.shape[0], 1))], axis=-1
        )
        img_keypoints = visualizer.draw_skeleton(img_keypoints, keypoints_2d)

    # Combine all meshes
    all_pred_vertices = []
    all_faces = []
    for pid, person_output in enumerate(outputs_sorted):
        # Apply the same left shift to each person's vertices before combining
        shifted_vertices = person_output["pred_vertices"] + person_output["pred_cam_t"]
        shifted_vertices[:, 0] -= LEFT_SHIFT_AMOUNT  # Shift X left
        all_pred_vertices.append(shifted_vertices)
        all_faces.append(faces + len(person_output["pred_vertices"]) * pid)

    all_pred_vertices = np.concatenate(all_pred_vertices, axis=0)
    all_faces = np.concatenate(all_faces, axis=0)

    # Compute fake camera translation (center of closest meshes)
    fake_pred_cam_t = (np.max(all_pred_vertices[-2*18439:], axis=0) + np.min(all_pred_vertices[-2*18439:], axis=0)) / 2
    all_pred_vertices = all_pred_vertices - fake_pred_cam_t

    # Use any person's focal length (they should be similar)
    sample_person = outputs_sorted[0]
    renderer = Renderer(focal_length=1.5 * sample_person["focal_length"], faces=all_faces)

    # Front view overlay
    img_mesh = (
        renderer(
            all_pred_vertices,
            fake_pred_cam_t,
            img_mesh,
            mesh_base_color=LIGHT_BLUE,
            scene_bg_color=(1, 1, 1),
        )
        * 255
    )

    # Side view
    white_img = np.ones_like(img_cv2) * 255
    img_mesh_side = (
        renderer(
            all_pred_vertices,
            fake_pred_cam_t,
            white_img,
            mesh_base_color=LIGHT_BLUE,
            scene_bg_color=(1, 1, 1),
            side_view=True,
        )
        * 255
    )

    cur_img = np.concatenate([img_cv2, img_keypoints, img_mesh, img_mesh_side], axis=1)
    return cur_img

In [ ]:
ls

In [ ]:
cd /kaggle/working/sam-3d-body

In [ ]:
from notebook.utils import setup_sam_3d_body
from tools.vis_utils import visualize_sample_together
import cv2
import numpy as np

estimator = setup_sam_3d_body(hf_repo_id="facebook/sam-3d-body-dinov3")
img_bgr = cv2.imread("/kaggle/input/ggggggg/eee.jpeg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

outputs = estimator.process_one_image(img_rgb)
rend_img = visualize_sample_together(img_bgr, outputs, estimator.faces)

cv2.imwrite("output.jpg", rend_img.astype(np.uint8))

In [ ]:
from IPython.display import Image, display

# Display the image
display(Image('/kaggle/working/sam-3d-body/output.jpg'))